# vdb_embedding_colab.ipynb — KOSIS 표 28만여 개를 벡터DB(Chroma)용으로 임베딩

배경: `agent/kosis/crawl_table_catalog.py`로 KOSIS 통계표 전체(약 28만7천개)의
ID/이름/기관코드를 크롤링해뒀다(`agent/kosis/crawl_output/tables.jsonl`). 카탈로그
매칭(3단계)에서 지금 쓰는 64개 수동 카탈로그만으로는 커버리지가 부족해서, 이 28만개를
의미 기반 검색(임베딩)으로 보조 후보를 찾는 데 쓰려고 한다.

이만한 양을 로컬(RAM 7.4GB)에서 임베딩하면 위험해서, 기존 리랭커/임베딩 작업과 같은
방식으로 코랩에서 배치 처리한다. `agent/kosis/prepare_vdb_export.py`가 만든
`data/vdb_pending.jsonl`(표 ID + 이름)을 여기서 읽어서 e5-large로 임베딩하고,
결과(벡터 + ID)를 다시 로컬로 가져가서 Chroma에 적재한다.

## 사용법
1. 로컬에서 `python -m agent.kosis.prepare_vdb_export` 실행 → `data/vdb_pending.jsonl` 생성
2. 그 파일을 구글 드라이브 "내 드라이브" 최상위에 업로드
3. VSCode에서 이 노트북을 코랩 커널에 연결 (우측 상단 커널 선택 → Colab)
4. 아래 셀을 순서대로 실행
5. 마지막 셀이 끝나면 구글 드라이브에 저장된 결과 파일 2개를 로컬 `data/`로 받아서
   `python -m agent.kosis.build_vdb_index`(다음 단계, 아직 안 만듦)로 Chroma에 적재

## 1. 연결 확인

In [ ]:
import platform
print("플랫폼:", platform.platform())
try:
    import google.colab  # noqa: F401
    print("코랩에서 실행 중")
except ImportError:
    print("코랩 아님 — 로컬에서 도는 중 (커널 선택이 안 된 상태일 수 있음)")

!cat /proc/meminfo 2>/dev/null | head -3 || echo "(Linux 환경 아님)"

## 2. 드라이브 마운트 + 데이터 로드

`data/vdb_pending.jsonl`을 구글 드라이브 "내 드라이브" 최상위에 미리 업로드해두세요.

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

SRC = "/content/drive/MyDrive/vdb_pending.jsonl"
pending_filename = "vdb_pending.jsonl"
shutil.copy(SRC, pending_filename)
print("복사됨:", pending_filename)

In [ ]:
import json

rows = []
with open(pending_filename, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"표 {len(rows)}건 로드됨")
print("샘플:", rows[0])

## 3. 임베딩 생성

64개 카탈로그 때와 같은 모델(intfloat/multilingual-e5-large)을 쓴다 — 나중에 claim
매칭 때 이 벡터랑 비교할 claim 쪽 임베딩도 같은 모델로 만들어야 서로 비교가 되므로
반드시 통일해야 한다. e5 계열은 "passage: " 접두사를 붙여야 문서 임베딩 성능이
제대로 나온다(공식 권장 사항).

28만여 건이라 시간이 좀 걸릴 수 있다(T4 기준 대략 수십 분 예상) — `show_progress_bar=True`로
진행 상황을 볼 수 있다.

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("intfloat/multilingual-e5-large")
print("모델 로드 완료")

In [ ]:
import numpy as np
import time

texts = ["passage: " + r["text"] for r in rows]

t0 = time.time()
embeddings = embed_model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"임베딩 완료: {embeddings.shape}, {time.time() - t0:.1f}초 소요")

## 4. 결과 저장 (드라이브로)

벡터는 JSON이 아니라 numpy 바이너리(.npy)로 저장한다 — 28만7천 × 1024차원을 텍스트로
저장하면 용량이 훨씬 커지고 느리다. ID/기관코드/원문은 임베딩 배열과 같은 순서로 저장된
별도 jsonl로 남겨서, 나중에 로컬에서 인덱스(줄 번호)로 다시 짝지을 수 있게 한다.

`files.download()`는 이 VS Code<->코랩 연결에서 실제 다운로드가 안 뜨는 문제가 있어서
(리랭커 노트북 때와 동일), 마운트된 드라이브에 직접 저장하는 방식으로 우회한다.

In [ ]:
np.save("vdb_embeddings.npy", embeddings.astype(np.float32))

with open("vdb_metadata.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps({"tbl_id": r["tbl_id"], "org_id": r.get("org_id"), "text": r["text"]}, ensure_ascii=False) + "\n")

shutil.copy("vdb_embeddings.npy", "/content/drive/MyDrive/vdb_embeddings.npy")
shutil.copy("vdb_metadata.jsonl", "/content/drive/MyDrive/vdb_metadata.jsonl")

print("저장 완료: /content/drive/MyDrive/vdb_embeddings.npy, vdb_metadata.jsonl")
print("두 파일을 로컬 data/ 폴더로 받아서 build_vdb_index.py를 실행하세요.")